# RaCo + ALIKED + LightGlue matching demo
Port of [cvg/RaCo `matching_demo.ipynb`](https://github.com/cvg/RaCo/blob/main/matching_demo.ipynb) at commit `ee06fa68`. RaCo weights and the sample images come from `@RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS`. LightGlue downloads its own ALIKED and matcher weights at runtime, as upstream does.

In [ ]:
# Pin numpy so OpenCV doesn't upgrade it past the preinstalled scipy/numba
%pip install --quiet kornia opencv-python-headless "numpy<2.4"
%pip install --quiet --no-deps git+https://github.com/cvg/RaCo.git@ee06fa68dbb20303f334fbc257dad71d4149603b
# install LightGlue from source
%pip install --quiet --no-deps git+https://github.com/cvg/LightGlue.git

In [ ]:
from pathlib import Path
from snowflake.snowpark.context import get_active_session

session = get_active_session()
STAGE = '@RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS'
ASSET_DIR = Path('/tmp/raco_demo')
(ASSET_DIR / 'assets').mkdir(parents=True, exist_ok=True)
session.file.get(f'{STAGE}/raco.pth', str(ASSET_DIR))
for name in ['v_grace_1.ppm', 'v_grace_3.ppm']:
    session.file.get(f'{STAGE}/assets/{name}', str(ASSET_DIR / 'assets'))
WEIGHTS = str(ASSET_DIR / 'raco.pth')
print(sorted(p.name for p in ASSET_DIR.rglob('*') if p.is_file()))

In [ ]:
import torch
from lightglue import ALIKED, LightGlue
from raco import RaCo
from raco.utils import load_image, rbd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize the models
extractor = RaCo(max_num_keypoints=1024, weights=WEIGHTS).eval().to(device)
descriptor = ALIKED().eval().to(device)
matcher = LightGlue(features="raco-aliked").eval().to(device)

In [ ]:
image0 = load_image(ASSET_DIR / "assets/v_grace_1.ppm").to(device)
image1 = load_image(ASSET_DIR / "assets/v_grace_3.ppm").to(device)

# Extract and describe keypoints
with torch.no_grad():
    feats0 = extractor.extract(image0)
    feats0["descriptors"] = descriptor.describe(feats0["keypoints"], image0)
    feats1 = extractor.extract(image1)
    feats1["descriptors"] = descriptor.describe(feats1["keypoints"], image1)

    # Match keypoints
    matches01 = matcher({"image0": feats0, "image1": feats1})
feats0, feats1, matches01 = [
    rbd(x) for x in [feats0, feats1, matches01]
]  # remove batch dimension

# Extract matches
kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]

In [ ]:
import matplotlib.pyplot as plt
from lightglue import viz2d

viz2d.plot_images([image0, image1])
viz2d.plot_matches(m_kpts0, m_kpts1, color=None, lw=0.2)
viz2d.add_text(0, f"Stop after {matches01['stop']} layers", fs=20)
viz2d.add_text(1, f"{len(m_kpts0)} Matches", fs=20)
plt.gcf().subplots_adjust(top=0.88)
plt.suptitle("RaCo Keypoints + ALIKED Descriptor + LightGlue+ Matches", fontsize=20)

viz2d.plot_images([image0, image1])
viz2d.plot_keypoints([kpts0, kpts1], colors="red")
viz2d.plot_keypoints([m_kpts0, m_kpts1], colors="lime")
plt.legend(["Unmatched", "Matched"], loc="upper right")
plt.gcf().subplots_adjust(top=0.88)
plt.suptitle("RaCo Keypoints", fontsize=20)
plt.show()